# 02 — Satellite azimuthal-anisotropy fits: TNG100 ($>10^8$) & TNG50 ($>10^7$)

Reads the MW-mass-host catalogs from `01_generate_catalogs.ipynb` (in `../data2`) and
fits the **azimuthal anisotropy** of the satellite distribution,
$p(\theta)\propto 1 + A\cos 2\theta$ on $\theta\in[0,90]$ ($0^\circ$ = host major axis). $A>0$ is a
major-axis excess; $A=0$ is isotropic. The amplitude $A$ is sampled with `emcee` (unbinned
likelihood, same as `notebooks/16`).

**What is plotted (two panels):**
* **Left — TNG100**, satellites $M_{*,\rm sat} > 10^8\,M_\odot$, at **z = 0** and **z = 0.05**.
* **Right — TNG50**, satellites $M_{*,\rm sat} > 10^7\,M_\odot$, at **z = 0** and **z = 0.05**.

It then also computes the **satellite quench fraction vs angle**, $f_q(\theta)=a+b\cos2\theta$,
with the same MCMC machinery and the same cuts (second figure, lower down).

**Radius cut toggle.** `RADIUS_CUT = False` by default → **no $R_{200c}$ cut** (all radii). Set it
to `True` to keep only satellites within `R200C_FACTOR` $\times R_{200c}$ (3D). Because notebook 01
writes satellites at all radii and stores `d_r200_3d`, this is a pure post-selection — no
regeneration needed. The toggle applies to **both** the anisotropy and quench-fraction figures.

> "z = 0.5" is interpreted as **z = 0.05** (snap 98), consistent with notebook 01. Missing catalogs
> are skipped with a warning, so this runs on whatever `../data2` currently holds.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import emcee
import matplotlib as mpl
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
%matplotlib inline
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
})

# --- configuration -----------------------------------------------------------
DATA_ROOT = "../data2"
CAT_TAG   = "logM7.00"                     # the mass floor notebook 01 wrote (satellites > 1e7)
CAT_NAME  = f"tng_satellites_hostlogM12.0-12.5_{CAT_TAG}.csv"   # must match notebook 01's HOST_TAG

# radius-cut toggle: False = no R200c cut (all radii); True = keep d_r200_3d < R200C_FACTOR
RADIUS_CUT   = False
R200C_FACTOR = 1.0

# what each panel shows: (sim, satellite log10 M* cut, panel color)
PANELS = [
    ("tng100", 8.0, "#1f77b4"),           # TNG100: M*_sat > 1e8
    ("tng50",  7.0, "#d62728"),           # TNG50:  M*_sat > 1e7
]
# redshifts overlaid within each panel: (subdir, label, linestyle, marker)
REDS = [("z0", "z = 0", "-", "o"), ("z0p05", "z = 0.05", "--", "s")]

# 18 angle bins over [0, 90]
N_BINS        = 18
ANGLE_EDGES   = np.linspace(0, 90, N_BINS + 1)
ANGLE_CENTERS = 0.5 * (ANGLE_EDGES[:-1] + ANGLE_EDGES[1:])

print(f"radius cut: {'d_r200_3d < %.2f R200c' % R200C_FACTOR if RADIUS_CUT else 'NONE (all radii)'}")

## Helpers — $P(\theta)$, the model curve, and the unbinned MCMC $A$ fit

Same anisotropy machinery as `notebooks/16`.

In [ ]:
def norm_hist(theta):
    c, _ = np.histogram(theta, bins=ANGLE_EDGES)
    return c / c.sum() / (90.0 / N_BINS)

def model_curve(A):
    x = np.linspace(0, 90, 200)
    return x, (1.0 / 90.0) * (1.0 + A * np.cos(2 * np.radians(x)))

def fit_anisotropy(theta_deg, n_walkers=16, n_steps=4000, burn=1000, seed=0):
    '''MCMC of p(theta) ~ 1 + A cos(2 theta). Returns dict with A median/std/percentiles.'''
    th = np.radians(np.asarray(theta_deg, dtype=float))
    c2 = np.cos(2 * th); n = len(th)
    if n < 5:
        return dict(n=n, A=np.nan, Aerr=np.nan, lo=np.nan, hi=np.nan)

    def log_prob(p):
        A = p[0]
        if not (-0.999 < A < 0.999):
            return -np.inf
        v = 1.0 + A * c2
        if np.any(v <= 0):
            return -np.inf
        return np.sum(np.log(v))

    rng = np.random.default_rng(seed)
    p0 = rng.uniform(-0.1, 0.1, size=(n_walkers, 1))
    sampler = emcee.EnsembleSampler(n_walkers, 1, log_prob)
    sampler.run_mcmc(p0, n_steps, progress=False)
    chain = sampler.get_chain(discard=burn, flat=True)[:, 0]
    lo, med, hi = np.percentile(chain, [16, 50, 84])
    return dict(n=n, A=med, Aerr=chain.std(), lo=lo, hi=hi)

# ---- quench fraction f_q = a + b cos(2 theta) (same model as notebooks 03/06/16) ----
def bootstrap_fq(angle, quenched, N=5000, seed=0):
    '''Binned-mean quench fraction per angle bin, with bootstrap mean and 1-sigma error.'''
    rng = np.random.default_rng(seed)
    angle = np.asarray(angle); quenched = np.asarray(quenched, dtype=float)
    n = len(angle)
    boot = np.full((N, N_BINS), np.nan)
    bin_idx = np.digitize(angle, ANGLE_EDGES) - 1
    for i in range(N):
        s = rng.integers(0, n, n)
        bi, qi = bin_idx[s], quenched[s]
        for j in range(N_BINS):
            m = bi == j
            if m.any():
                boot[i, j] = qi[m].mean()
    return np.nanmean(boot, axis=0), np.nanstd(boot, axis=0)

def _fq_log_prob(theta, x, y, sigma):
    a, b, f = theta
    if not (0 < a < 1 and -1 < b < 1 and -10 < f < 2):
        return -np.inf
    s = sigma ** 2 + np.exp(f) ** 2
    model = a + b * np.cos(2 * np.radians(x))
    return -0.5 * np.sum((y - model) ** 2 / s + np.log(2 * np.pi * s))

def fit_fq_sinusoid(mean, std, n_walkers=20, n_steps=6000, burn=1000, seed=0):
    '''MCMC fit of f_q = a + b cos(2 theta); returns (params_mean, params_std).'''
    np.random.seed(seed)
    ok = np.isfinite(mean) & np.isfinite(std) & (std > 0)
    if ok.sum() < 4:
        return np.array([np.nan] * 3), np.array([np.nan] * 3)
    p0 = np.array([0.5, 0.0, -3.0]) + 1e-2 * np.random.randn(n_walkers, 3)
    sampler = emcee.EnsembleSampler(n_walkers, 3, _fq_log_prob,
                                    args=(ANGLE_CENTERS[ok], mean[ok], std[ok]))
    sampler.run_mcmc(p0, n_steps, progress=False)
    chain = sampler.get_chain(discard=burn, flat=True)
    return chain.mean(axis=0), chain.std(axis=0)

def fq_band(p, e, n_mc=5000, seed=0):
    '''Posterior median + 16/84th-percentile band of f_q = a + b cos(2 theta) over [0, 90].'''
    rng = np.random.default_rng(seed)
    x = np.linspace(0, np.pi / 2, 300)
    a = rng.normal(p[0], e[0], n_mc); b = rng.normal(p[1], e[1], n_mc)
    yy = a[:, None] + b[:, None] * np.cos(2 * x)[None, :]
    return np.degrees(x), a.mean() + b.mean() * np.cos(2 * x), np.percentile(yy, 16, 0), np.percentile(yy, 84, 0)

## Load the catalogs and apply the satellite mass cut (and optional radius cut)

For each panel we read the single all-central catalog per redshift, keep `mstar_phys > cut`, and —
if `RADIUS_CUT` — also `d_r200_3d < R200C_FACTOR`. Missing files are skipped with a warning.

In [ ]:
def load_sat(sim, zsub, logcut):
    '''Return (alpha, quenched) arrays after the mass cut and optional radius cut; None if missing.'''
    path = os.path.join(DATA_ROOT, sim, zsub, CAT_NAME)
    if not os.path.exists(path):
        print(f"[skip] missing {path}")
        return None
    df = pd.read_csv(path)
    df = df[df["mstar_phys"] > logcut]
    if RADIUS_CUT:
        df = df[df["d_r200_3d"] < R200C_FACTOR]
    return df["alpha"].to_numpy(), df["quenched"].to_numpy(dtype=float)

data, quench, fits = {}, {}, {}
for sim, logcut, _ in PANELS:
    for zsub, zlabel, _, _ in REDS:
        key = (sim, zsub)
        res = load_sat(sim, zsub, logcut)
        if res is None:
            continue
        th, q = res
        if len(th) == 0:
            continue
        data[key], quench[key] = th, q
        fits[key] = fit_anisotropy(th, seed=1)

print(f"{'dataset':<18s} {'N':>6s}  {'A':>16s}   |A|/sig")
for sim, logcut, _ in PANELS:
    for zsub, zlabel, _, _ in REDS:
        key = (sim, zsub)
        if key not in fits:
            continue
        f = fits[key]
        sig = abs(f["A"] / f["Aerr"]) if np.isfinite(f["A"]) else np.nan
        print(f"{sim+' '+zlabel:<18s} {f['n']:>6d}  {f['A']:+.3f} +/- {f['Aerr']:.3f}   {sig:.2f}")

## Figure — TNG100 ($>10^8$) vs TNG50 ($>10^7$), each at z=0 and z=0.05

Stepped histogram = $P(\theta)$; smooth curve = posterior-median $\tfrac{1}{90}(1+A\cos2\theta)$;
dotted line = isotropic level. Solid = z=0, dashed = z=0.05.

In [ ]:
SATLABEL = {8.0: r"$M_* > 10^8\,M_\odot$", 7.0: r"$M_* > 10^7\,M_\odot$"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (sim, logcut, color) in zip(axes, PANELS):
    for zsub, zlabel, ls, marker in REDS:
        key = (sim, zsub)
        if key not in fits:
            continue
        f = fits[key]
        h = norm_hist(data[key])
        ax.step(ANGLE_EDGES, np.r_[h, h[-1]], where="post", color=color, ls=ls, lw=1.5,
                alpha=0.6, label=f"{zlabel}  (N={f['n']}, A={f['A']:+.2f})")
        if np.isfinite(f["A"]):
            x, y = model_curve(f["A"]); ax.plot(x, y, color=color, ls=ls, lw=2.5)
    ax.axhline(1 / 90, color="k", lw=1, ls=":")
    ax.set_xlim(0, 90); ax.set_ylim(0, 0.025)
    ax.set_xlabel(r"$\theta$ [deg]")
    ax.set_title(f"{sim.upper()}   ({SATLABEL[logcut]})")
    ax.legend(fontsize=9, fancybox=False, edgecolor="k")
    ax.tick_params(which="both", direction="in", top=True, right=True)
axes[0].set_ylabel(r"$P(\theta)$")
_rc = f"within {R200C_FACTOR:g} R200c" if RADIUS_CUT else "no radius cut"
fig.suptitle(f"Satellite azimuthal anisotropy  (MW-mass hosts 12.0-12.5, {_rc})", y=1.02, fontsize=14)
plt.subplots_adjust(wspace=0.08)
plt.show()

## Quench fraction vs angle $f_q(\theta)=a+b\cos2\theta$

Same datasets and cuts as above (TNG100 $>10^8$, TNG50 $>10^7$, each at z=0 and z=0.05, honoring
the `RADIUS_CUT` toggle), now for the **satellite quench fraction** vs angle. The quench flag is
the TNG `quenched` column (Martín-Navarro+ 2021 SFMS). We bootstrap $f_q$ in angle bins and fit
$f_q(\theta)=a+b\cos2\theta$ with `emcee` (same model as notebooks 03/06/16); $b=0$ is isotropic
quenching, $b>0$ = quenching enhanced toward the host major axis.

In [ ]:
fq_mean, fq_std, fq_p, fq_e = {}, {}, {}, {}
for sim, logcut, _ in PANELS:
    for zsub, zlabel, _, _ in REDS:
        key = (sim, zsub)
        if key not in data:
            continue
        m, s = bootstrap_fq(data[key], quench[key])
        fq_mean[key], fq_std[key] = m, s
        fq_p[key], fq_e[key] = fit_fq_sinusoid(m, s)

print(f"{'dataset':<18s}  {'a':>16s}   {'b':>16s}   |b|/sig_b")
for sim, logcut, _ in PANELS:
    for zsub, zlabel, _, _ in REDS:
        key = (sim, zsub)
        if key not in fq_p:
            continue
        p, e = fq_p[key], fq_e[key]
        sig = abs(p[1] / e[1]) if np.isfinite(p[1]) else np.nan
        print(f"{sim+' '+zlabel:<18s}  {p[0]:.3f} +/- {e[0]:.3f}   {p[1]:+.3f} +/- {e[1]:.3f}   {sig:.2f}")

### Figure — quench fraction: TNG100 ($>10^8$) vs TNG50 ($>10^7$), z=0 and z=0.05

Points + error bars = bootstrap $f_q$ per angle bin; line + band = MCMC posterior median and
16-84th percentile. Solid/circles = z=0, dashed/squares = z=0.05.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (sim, logcut, color) in zip(axes, PANELS):
    for zsub, zlabel, ls, marker in REDS:
        key = (sim, zsub)
        if key not in fq_p:
            continue
        m, s = fq_mean[key], fq_std[key]
        p, e = fq_p[key], fq_e[key]
        sig = abs(p[1] / e[1]) if np.isfinite(p[1]) else np.nan
        ax.errorbar(ANGLE_CENTERS, m, yerr=s, fmt=marker, color=color, capsize=3, ms=5, ls="none",
                    label=f"{zlabel}  (b={p[1]:+.3f}, |b|/$\\sigma$={sig:.1f})")
        if np.isfinite(p[1]):
            x, ymed, ylo, yhi = fq_band(p, e)
            ax.plot(x, ymed, color=color, ls=ls, lw=2)
            ax.fill_between(x, ylo, yhi, color=color, alpha=0.12)
    ax.set_xlim(0, 90); ax.set_ylim(0, 1)
    ax.set_xlabel(r"$\theta$ [deg]")
    ax.set_title(f"{sim.upper()}   ({SATLABEL[logcut]})")
    ax.legend(fontsize=9, fancybox=False, edgecolor="k")
    ax.tick_params(which="both", direction="in", top=True, right=True)
axes[0].set_ylabel(r"$f_q$")
fig.suptitle(rf"Quench fraction vs angle ($f_q=a+b\cos2\theta$)  "
             rf"(MW-mass hosts 12.0-12.5, {_rc})", y=1.02, fontsize=14)
plt.subplots_adjust(wspace=0.08)
plt.show()